# Recursive Multi-Step — Random Forest

**Một** model học 1 bước (`row(tau) -> sales(tau+1)`), lăn 10 lần: dự đoán bước trước nạp lại làm lag cho bước sau.

Đứng ở tuần cuối train (2012-03-23), dự báo 10 tuần đầu val (2012-03-30 -> 2012-06-01). 45 store x 10 tuần = 450 dự đoán.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

In [ ]:
def get_file_path(filename):
    current_dir = Path.cwd()
    for search_root in [current_dir] + list(current_dir.parents):
        for file in search_root.rglob(filename):
            if file.is_file():
                return file
    raise FileNotFoundError(f"Không tìm thấy file {filename}!")


train_df = pd.read_csv(get_file_path("train_final.csv"))
val_df = pd.read_csv(get_file_path("val_set.csv"))

for df in (train_df, val_df):
    df["Date"] = pd.to_datetime(df["Date"])

train_set = train_df.sort_values(["Store", "Date"]).reset_index(drop=True)
val_set = val_df.sort_values(["Store", "Date"]).reset_index(drop=True)

print("train :", train_set.shape, "|", train_set["Date"].min().date(), "->", train_set["Date"].max().date())
print("val   :", val_set.shape, "|", val_set["Date"].min().date(), "->", val_set["Date"].max().date())

In [ ]:
HORIZON = 10
feature_cols = [c for c in train_set.columns if c not in ["Date", "Weekly_Sales", "Type"]]
print(f"{len(feature_cols)} cột feature")

# model 1 buoc: row(tau) -> sales(tau+1). Cung tham so voi ban direct de so duoc.
rf_params = dict(n_estimators=200, max_depth=12, min_samples_split=5,
                 min_samples_leaf=2, n_jobs=-1, random_state=42)

train_1step = train_set.copy()
train_1step["y"] = train_1step.groupby("Store")["Weekly_Sales"].shift(-1)
train_1step = train_1step.dropna(subset=["y"])

model = RandomForestRegressor(**rf_params).fit(train_1step[feature_cols], train_1step["y"])
print(f"train trên {len(train_1step)} dòng")

## Vòng lăn

Model học `row(tau) -> sales(tau+1)`: hàng feature đứng **trước nhãn một tuần**. Nên muốn dự báo tuần `w`, phải đưa vào hàng của tuần `w-1`, không phải hàng của tuần `w`. Đưa nhầm hàng `w` thì `Lag_1` cách nhãn 1 tuần thay vì 2 — model được ăn thông tin tươi hơn lúc train, sai số thấp giả tạo.

Lịch sử `hist` mồi bằng doanh số **thật** tới hết train. Mỗi bước: lấy hàng `w-1`, **ghi đè 5 cột phụ thuộc lịch sử** từ `hist`, dự đoán, nạp dự đoán vào `hist[w]`.

Hàng gốc mang sẵn `Lag_*` tính từ doanh số thật — không ghi đè thì model nhìn thấy tương lai.

| Cột | Cần tuần (so với hàng feature tau) | Nhiễm dự đoán từ bước |
|---|---|---|
| `Lag_1`, `Rolling_Mean_4w` | tau−1 .. tau−4 | k = 3 |
| `Lag_4` | tau−4 | k = 6 |
| `Lag_12`, `Lag_52` | tau−12, tau−52 | không bao giờ |

In [ ]:
LAST_TRAIN_WEEK = train_set["Date"].max()
W = lambda n: pd.Timedelta(weeks=n)

all_weeks = pd.concat([train_df, val_df], ignore_index=True)
sales_wide = all_weeks.pivot(index="Date", columns="Store", values="Weekly_Sales").sort_index()
STORES = sales_wide.columns.to_numpy()
# hang feature co the nam trong TRAIN (buoc 1) hoac trong VAL (cac buoc sau)
row_of = {d: g.sort_values("Store") for d, g in all_weeks.groupby("Date")}

# CHI moi bang doanh so that toi het train — moi tuan sau do phai la du doan
hist = {d: sales_wide.loc[d].to_numpy() for d in sales_wide.index if d <= LAST_TRAIN_WEEK}
print(f"mồi {len(hist)} tuần thật, tới {LAST_TRAIN_WEEK.date()}")


def wk(w, n):
    key = w - W(n)
    assert key in hist, f"thiếu lịch sử tuần {key.date()} (cần cho tuần {w.date()})"
    return hist[key]


rows = []
for k in range(1, HORIZON + 1):
    w = LAST_TRAIN_WEEK + W(k)      # tuần ĐÍCH
    tau = w - W(1)                  # hàng FEATURE đứng trước nhãn 1 tuần
    src = row_of[tau]
    assert np.array_equal(src["Store"].to_numpy(), STORES), "thứ tự Store lệch"

    X = src[feature_cols].copy()
    X["Lag_1"] = wk(tau, 1)
    X["Lag_4"] = wk(tau, 4)
    X["Lag_12"] = wk(tau, 12)
    X["Lag_52"] = wk(tau, 52)
    X["Rolling_Mean_4w"] = np.mean([wk(tau, i) for i in (1, 2, 3, 4)], axis=0)

    preds = model.predict(X)
    hist[w] = preds

    rows.append(pd.DataFrame({
        "Store": STORES,
        "Date": LAST_TRAIN_WEEK,
        "horizon": k,
        "target_Date": w,
        "y_true": row_of[w]["Weekly_Sales"].to_numpy(),
        "y_pred": preds,
    }))

predictions = pd.concat(rows, ignore_index=True)

# chot: buoc 1 chi dung so that -> phai trung khit du doan tu hang cuoi train
assert np.allclose(predictions.query("horizon == 1")["y_pred"],
                   model.predict(row_of[LAST_TRAIN_WEEK][feature_cols])), "bước 1 sai"
print(f"{len(predictions)} dòng dự đoán ({HORIZON} tuần x {len(STORES)} store)")

In [ ]:
recs = []
for k in range(1, HORIZON + 1):
    s = predictions[predictions["horizon"] == k]
    y, p = s["y_true"], s["y_pred"]
    recs.append({
        "Bước": f"t+{k:02d}",
        "Tuần": s["target_Date"].iloc[0].date(),
        "RMSE": f"{root_mean_squared_error(y, p):,.0f}",
        "WAPE": f"{np.abs(y - p).sum() / np.abs(y).sum() * 100:.2f}%",
        "MAE": f"{mean_absolute_error(y, p):,.0f}",
    })

display(pd.DataFrame(recs).set_index("Bước"))

y, p = predictions["y_true"], predictions["y_pred"]
print(f"Gộp cả {HORIZON} tuần | RMSE {root_mean_squared_error(y, p):,.0f}"
      f" | WAPE {np.abs(y - p).sum() / np.abs(y).sum():.2%}"
      f" | MAE {mean_absolute_error(y, p):,.0f}")

In [ ]:
OUT = Path.cwd() / "predictions_recursive.csv"
predictions.to_csv(OUT, index=False)
print("đã lưu:", OUT, "|", predictions.shape)